# SQL Avanzado

### Introducción

En esta actividad vamos a profundizar en dos de las herramientas más poderosas de SQL: las **subconsultas** y las **window functions**. Ambas permiten escribir consultas mucho más expresivas que las que vimos anteriormente.

### Esquema

Seguimos trabajando con el mismo esquema de siempre:

- `Capitanes(cid INT PRIMARY KEY, cnombre VARCHAR(100), crating FLOAT, cedad INT)`
- `Botes(bid INT PRIMARY KEY, bnombre VARCHAR(100), bcolor VARCHAR(100))`
- `Reservas(cid INT, bid INT, fecha DATE, PRIMARY KEY(cid, bid))`

### Outline

En esta actividad aprenderemos:

- Subconsultas en `WHERE`, `FROM` y `SELECT`.
- Subconsultas correlacionadas y el operador `EXISTS`.
- Window functions: `OVER`, `PARTITION BY`, `ORDER BY`.
- Funciones de ranking: `ROW_NUMBER`, `RANK`, `DENSE_RANK`.
- Funciones de navegación: `LAG` y `LEAD`.
- Agregados acumulados con frames.

## Setup

Primero creamos la base de datos y la llenamos con los datos del esquema. **Ejecuta esta celda antes de cualquier otra.**

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")  # base de datos en memoria, se resetea al cerrar
cursor = conn.cursor()

# --- Crear tablas ---
cursor.executescript("""
    DROP TABLE IF EXISTS Reservas;
    DROP TABLE IF EXISTS Capitanes;
    DROP TABLE IF EXISTS Botes;

    CREATE TABLE Capitanes(
        cid     INT PRIMARY KEY,
        cnombre VARCHAR(100),
        crating FLOAT DEFAULT 0,
        cedad   INT
    );

    CREATE TABLE Botes(
        bid    INT PRIMARY KEY,
        bnombre VARCHAR(100),
        bcolor  VARCHAR(100)
    );

    CREATE TABLE Reservas(
        cid  INT,
        bid  INT,
        fecha DATE,
        PRIMARY KEY(cid, bid)
    );

    -- Capitanes
    INSERT INTO Capitanes VALUES(23, 'King Arturo', 8,   31);
    INSERT INTO Capitanes VALUES(29, 'Juan',        1,   33);
    INSERT INTO Capitanes VALUES(31, 'Andy',        8,   55);
    INSERT INTO Capitanes VALUES(32, 'Felipe',      8.4, 25);
    INSERT INTO Capitanes VALUES(58, 'Oscar',       10,  35);
    INSERT INTO Capitanes VALUES(64, 'Isidora',     7.5, 35);
    INSERT INTO Capitanes VALUES(71, 'Pedro',       10,  16);
    INSERT INTO Capitanes VALUES(74, 'Isidora',     9,   35);
    INSERT INTO Capitanes VALUES(85, 'Rosa',        3,   25);
    INSERT INTO Capitanes VALUES(95, 'Romano',      5.5, 63);

    -- Botes
    INSERT INTO Botes VALUES(101, 'Catamaran', 'Azul');
    INSERT INTO Botes VALUES(102, 'Catamaran', 'Rojo');
    INSERT INTO Botes VALUES(103, 'Endurance', 'Verde');
    INSERT INTO Botes VALUES(104, 'Yate',      'Rojo');

    -- Reservas
    INSERT INTO Reservas VALUES(23, 101, '2016-10-10');
    INSERT INTO Reservas VALUES(23, 102, '2016-10-10');
    INSERT INTO Reservas VALUES(23, 103, '2016-10-08');
    INSERT INTO Reservas VALUES(23, 104, '2017-10-07');
    INSERT INTO Reservas VALUES(31, 102, '2017-11-10');
    INSERT INTO Reservas VALUES(31, 103, '2018-11-06');
    INSERT INTO Reservas VALUES(31, 104, '2018-11-12');
    INSERT INTO Reservas VALUES(64, 101, '2018-09-05');
    INSERT INTO Reservas VALUES(64, 102, '2018-09-08');
    INSERT INTO Reservas VALUES(74, 103, '2018-09-08');
""")
conn.commit()
print("Base de datos lista!")

In [ ]:
def print_rows(cursor, rows):
    if not rows:
        print("(sin resultados)")
        return
    cols = [d[0] for d in cursor.description]
    widths = [max(len(str(x)) for x in col) for col in zip(cols, *rows)]
    header = " | ".join(f"{c:<{w}}" for c, w in zip(cols, widths))
    print(header)
    print("-" * len(header))
    for row in rows:
        print(" | ".join(f"{str(v):<{w}}" for v, w in zip(row, widths)))

def query(sql):
    """Ejecuta una consulta y muestra el resultado formateado."""
    cursor.execute(sql)
    rows = cursor.fetchall()
    print_rows(cursor, rows)

---
## Parte 1 — Subconsultas

Una subconsulta es simplemente una consulta `SELECT` anidada dentro de otra. Como cada consulta retorna una tabla, podemos usarla en tres lugares:

| Lugar | ¿Qué retorna la subconsulta? | Operadores útiles |
|---|---|---|
| `WHERE` | Un escalar o una columna | `=`, `IN`, `NOT IN`, `EXISTS` |
| `FROM` | Una tabla (hay que darle alias) | — |
| `SELECT` | Un escalar (una sola fila/columna) | — |

### 1.1 Subconsulta en WHERE — valor escalar

El caso más simple: la subconsulta retorna un único valor y lo usamos con `=`, `>`, `<`, etc.

**Ejemplo**: Nombres de los capitanes con el rating más alto.

In [ ]:
query("""
SELECT cnombre, crating
FROM   Capitanes
WHERE  crating = (SELECT MAX(crating) FROM Capitanes)
""")

**Ejercicio 1.** Encuentra los nombres y la edad de los capitanes cuya edad es **menor que el promedio** de edad de todos los capitanes.

In [ ]:
# Tu consulta acá
query("""

""")

**Ejercicio 2.** Encuentra los nombres de los capitanes con rating **mayor que el promedio de rating de los capitanes menores de 40 años**.

*Hint: la subconsulta debe filtrar con `WHERE` antes de calcular el promedio.*

In [ ]:
query("""

""")

### 1.2 Subconsulta en WHERE — con IN y NOT IN

Cuando la subconsulta retorna una **columna** (múltiples filas), usamos `IN` o `NOT IN`.

**Ejemplo**: Nombres de los capitanes que han reservado algún bote.

In [ ]:
query("""
SELECT cnombre
FROM   Capitanes
WHERE  cid IN (SELECT cid FROM Reservas)
""")

**Ejercicio 3.** Encuentra los nombres de los capitanes que **nunca** han hecho una reserva.

In [ ]:
query("""

""")

**Ejercicio 4.** Encuentra el nombre y color de los botes que **no han sido reservados por ningún capitán con rating menor a 5**.

*Hint: primero piensa qué botes SÍ fueron reservados por capitanes con rating < 5, luego niega.*

In [ ]:
query("""

""")

### 1.3 Subconsulta en FROM — tabla derivada

Podemos usar una consulta como si fuera una tabla en el `FROM`. **Siempre hay que darle un alias.**

**Ejemplo**: ¿Qué capitanes hicieron más de una reserva? Primero contamos reservas por capitán, luego filtramos.

In [ ]:
query("""
SELECT cnombre, total_reservas
FROM   Capitanes
JOIN   (
           SELECT cid, COUNT(*) AS total_reservas
           FROM   Reservas
           GROUP BY cid
       ) AS resumen ON Capitanes.cid = resumen.cid
WHERE  total_reservas > 1
""")

**Ejercicio 5.** Encuentra el nombre del capitán (o los capitanes) que ha hecho **la mayor cantidad de reservas**. Usa una subconsulta en el `FROM` para calcular el conteo por capitán, y otra subconsulta (o `MAX`) para encontrar el máximo.

In [ ]:
query("""

""")

**Ejercicio 6.** Para cada color de bote, muestra el color y la **edad promedio de los capitanes que reservaron ese color**. Usa una subconsulta en el `FROM` que haga el join entre Reservas, Botes y Capitanes para obtener el color y la edad de cada reserva, y luego agrupa en la consulta exterior.

*Resultado esperado: una fila por color (Azul, Rojo, Verde) con su promedio de edad.*

In [ ]:
query("""

""")

### 1.4 Subconsultas correlacionadas y EXISTS

Una subconsulta **correlacionada** referencia una columna de la consulta exterior. Se ejecuta una vez por cada fila de la consulta exterior.

`EXISTS` retorna `TRUE` si la subconsulta produce al menos una fila.

**Ejemplo**: Capitanes que han reservado **algún** bote rojo (usando `EXISTS`).

In [ ]:
query("""
SELECT cnombre
FROM   Capitanes c
WHERE  EXISTS (
           SELECT 1
           FROM   Reservas r
           JOIN   Botes b ON r.bid = b.bid
           WHERE  r.cid = c.cid        -- correlación: referencia c de afuera
           AND    b.bcolor = 'Rojo'
       )
""")

Nota que escribimos `SELECT 1` porque no nos importa qué retorna la subconsulta, solo si retorna algo.

**Ejercicio 7.** Usando `NOT EXISTS`, encuentra los capitanes que **nunca reservaron un bote verde**.

In [ ]:
query("""

""")

**Ejercicio 8 (difícil).** Encuentra los capitanes que han reservado **todos** los botes. 

*Hint: un capitán reservó todos los botes si no existe ningún bote que él NO haya reservado. Usa `NOT EXISTS` con dos subconsultas anidadas.*

In [ ]:
query("""

""")

### 1.5 Subconsulta en SELECT

También podemos poner una subconsulta directamente en el `SELECT`. Debe retornar **exactamente un valor** (una fila, una columna).

**Ejemplo**: Para cada capitán, muestra su nombre y la diferencia entre su rating y el rating promedio de todos.

In [ ]:
query("""
SELECT cnombre,
       crating,
       ROUND(crating - (SELECT AVG(crating) FROM Capitanes), 2) AS diff_promedio
FROM   Capitanes
ORDER BY diff_promedio DESC
""")

**Ejercicio 9.** Para cada bote, muestra su nombre, color y la **cantidad de veces que fue reservado** (usando una subconsulta correlacionada en el `SELECT`). Si nunca fue reservado, debe mostrar 0.

*Hint: usa `COALESCE(valor, 0)` para convertir NULL en 0.*

In [ ]:
query("""

""")

---
## Parte 2 — Window Functions

Las window functions calculan un valor para cada fila **basándose en un conjunto de filas relacionadas** (la "ventana"), sin colapsar las filas como lo hace `GROUP BY`.

```sql
funcion() OVER (
    PARTITION BY col   -- divide en grupos (como GROUP BY pero sin colapsar)
    ORDER BY     col   -- ordena dentro del grupo
)
```

**Idea clave**: `GROUP BY` reduce N filas a 1. Una window function sigue entregando N filas, pero cada fila ve información de su grupo.

### 2.1 Agregados sobre ventana

Todas las funciones de agregación (`SUM`, `AVG`, `MIN`, `MAX`, `COUNT`) funcionan con `OVER`.

**Ejemplo**: Para cada capitán, muestra su rating junto con el rating promedio de **todos** los capitanes y el de **su misma edad**.

In [ ]:
query("""
SELECT cnombre,
       cedad,
       crating,
       ROUND(AVG(crating) OVER (), 2)                    AS prom_global,
       ROUND(AVG(crating) OVER (PARTITION BY cedad), 2)  AS prom_misma_edad
FROM   Capitanes
ORDER BY cedad, cnombre
""")

**Ejercicio 10.** Muestra para cada reserva el `cid`, `bid`, la fecha, y el **total de reservas del mismo capitán** (cuántos botes distintos reservó ese capitán en total). Usa `COUNT(*) OVER (PARTITION BY ...)`.

*Resultado esperado: 10 filas (una por reserva), cada una con el conteo total de su capitán.*

In [ ]:
query("""

""")

**Ejercicio 11.** Para cada capitán, muestra su nombre, rating, y cuánto representa su rating como **porcentaje del rating total** de todos los capitanes. Redondea a 1 decimal.

*Fórmula: `ROUND(100.0 * crating / SUM(crating) OVER (), 1)`*

In [ ]:
query("""

""")

### 2.2 Window Functions vs GROUP BY

Veamos la diferencia concreta. Con `GROUP BY` colapsamos filas — perdemos el detalle individual. Con window functions conservamos todas las filas.

**Ejemplo**: Capitanes cuyo rating supera el promedio de su misma edad. Esto **no se puede hacer** con solo `GROUP BY` porque necesitamos comparar cada fila con el promedio de su grupo.

In [ ]:
# Paso 1: calculamos el promedio por edad con una WF en una subconsulta
query("""
SELECT cnombre, cedad, crating, prom_edad
FROM (
    SELECT cnombre,
           cedad,
           crating,
           ROUND(AVG(crating) OVER (PARTITION BY cedad), 2) AS prom_edad
    FROM Capitanes
) sub
WHERE crating > prom_edad
""")

¿Por qué necesitamos la subconsulta? Porque las window functions se evalúan en el `SELECT`, **después** del `WHERE`. No podemos filtrar con una WF directamente en el `WHERE` — hay que envolverla en una subconsulta.

**Ejercicio 12.** Encuentra los capitanes cuyo rating es **mayor o igual al rating máximo de su grupo de edad**.

*Hint: es el mismo patrón de arriba pero con `MAX` en vez de `AVG`.*

In [ ]:
query("""

""")

### 2.3 Funciones de Ranking

| Función | Empates | ¿Salta números? |
|---|---|---|
| `ROW_NUMBER()` | Número único (arbitrario) | — |
| `RANK()` | Mismo rango | Sí: 1, 1, 3 |
| `DENSE_RANK()` | Mismo rango | No: 1, 1, 2 |

**Ejemplo**: Ranking de capitanes por rating (de mayor a menor).

In [ ]:
query("""
SELECT cnombre,
       crating,
       ROW_NUMBER() OVER (ORDER BY crating DESC) AS row_num,
       RANK()       OVER (ORDER BY crating DESC) AS rk,
       DENSE_RANK() OVER (ORDER BY crating DESC) AS dense_rk
FROM   Capitanes
""")

Observa los capitanes con crating = 10 (Oscar y Pedro) y los con crating = 8 (King Arturo y Andy): ¿cómo difieren `RANK` y `DENSE_RANK`?

**Ejercicio 13.** Muestra el ranking de capitanes por rating **dentro de cada grupo de edad** (el mejor de cada edad tiene rango 1). Usa `DENSE_RANK`.

*Hint: agrega `PARTITION BY cedad` al `OVER`.*

In [ ]:
query("""

""")

**Ejercicio 14 — Patrón Top-N.** Muestra el capitán con **mejor rating por cada edad**. Si hay empate, muestra a todos.

*Hint: calcula el `DENSE_RANK` en una subconsulta y luego filtra `WHERE rk = 1` afuera.*

In [ ]:
query("""

""")

### 2.4 LAG y LEAD — Funciones de navegación

`LAG(expr)` accede al valor de la fila **anterior** dentro de la ventana.  
`LEAD(expr)` accede al valor de la fila **siguiente**.  
Si no existe la fila vecina, retornan `NULL`.

**Ejemplo**: Para cada reserva de un capitán (ordenadas por fecha), muestra la fecha de su reserva anterior.

In [ ]:
query("""
SELECT c.cnombre,
       r.bid,
       r.fecha,
       LAG(r.fecha)  OVER (PARTITION BY r.cid ORDER BY r.fecha) AS reserva_anterior,
       LEAD(r.fecha) OVER (PARTITION BY r.cid ORDER BY r.fecha) AS reserva_siguiente
FROM   Reservas r
JOIN   Capitanes c ON r.cid = c.cid
ORDER BY c.cnombre, r.fecha
""")

¿Por qué aparece `None` en la primera y última reserva de cada capitán?

**Ejercicio 15.** Para cada capitán que tiene más de una reserva, muestra cuántos días pasaron entre cada reserva y la anterior.

*Hint: En SQLite puedes restar fechas con `JULIANDAY(fecha) - JULIANDAY(LAG(fecha) OVER (...))`.*

In [ ]:
query("""

""")

**Ejercicio 16.** Para cada capitán, muestra su nombre, su rating, el rating del capitán que está **justo por encima** en el ranking general (el siguiente en `LEAD` si ordenas de mayor a menor) y la diferencia entre ellos.

*Resultado esperado: la fila del mejor capitán tendrá `None` en el rating siguiente.*

In [ ]:
query("""

""")

### 2.5 Acumulados y Frames

Cuando combinamos `OVER` con `ORDER BY`, el agregado se calcula de forma **acumulada** (desde el inicio de la partición hasta la fila actual). Esto es por el **frame implícito**:

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

Podemos cambiar el frame con `ROWS BETWEEN <inicio> AND <fin>`.

**Ejemplo**: Rating acumulado de capitanes ordenados por cid.

In [ ]:
query("""
SELECT cid,
       cnombre,
       crating,
       SUM(crating) OVER (ORDER BY cid) AS rating_acumulado
FROM   Capitanes
""")

**Ejercicio 17.** Muestra para cada capitán (ordenados por `cid`) su nombre, rating, y el **promedio móvil** de rating considerando él mismo más el capitán anterior y el siguiente.

*Hint: usa `AVG(crating) OVER (ORDER BY cid ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING)`.*

In [ ]:
query("""

""")

**Ejercicio 18.** Para cada reserva (ordenada por fecha), muestra el `cid`, `bid`, fecha, y la **cantidad acumulada de reservas** hasta esa fecha (inclusive).

*Hint: `COUNT(*) OVER (ORDER BY fecha ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`.*

In [ ]:
query("""

""")

---
## Parte 3 — Combinando Subconsultas y Window Functions

Los ejercicios más interesantes (y los que más salen en controles) mezclan ambas herramientas. La estructura típica es:

```sql
SELECT ...
FROM (
    SELECT ...,
           alguna_WF() OVER (...) AS resultado_wf
    FROM ...
) sub
WHERE <condición sobre resultado_wf>
```

**Ejercicio 19.** Encuentra los nombres de los capitanes cuyo rating es **mayor que el promedio de su misma edad**, y además han hecho al menos una reserva.

*Hint: combina la subconsulta en FROM con WF para el promedio por edad, y un IN/EXISTS para la condición de reserva.*

In [ ]:
query("""

""")

**Ejercicio 20.** Para cada capitán que ha hecho reservas, muestra su nombre y la **fecha de su reserva más reciente**. Luego ordena de más reciente a más antigua.

*Hint: puedes usar subconsulta en SELECT, o GROUP BY con MAX(fecha) + JOIN, o ROW_NUMBER() para quedarte con la última fila por capitán.*

In [ ]:
query("""

""")

**Ejercicio 21 (desafío).** Muestra una tabla con `cnombre`, `crating`, y una columna `posicion_global` que indica el puesto del capitán en el ranking general de rating. Además, muestra una columna `posicion_en_edad` con su puesto dentro de capitanes de su misma edad. Usa `RANK()` para ambas.

Ordena el resultado por `posicion_global`.

In [ ]:
query("""

""")

**Ejercicio 22 (desafío).** Encuentra los botes que fueron reservados por el capitán con **mejor ranking** (mayor rating). Si hay empate en rating, muéstralos todos.

*Hint: usa RANK() o DENSE_RANK() en una subconsulta para encontrar los capitanes con rango 1, luego busca sus reservas.*

In [ ]:
query("""

""")